In [7]:
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory

# Chemin d'accès au dossier contenant les images
images_dir = 'train/images'

# Chemin d'accès au dossier contenant les labels
labels_dir = 'train/labels'

# Charger les images à partir du dossier
images_dataset = image_dataset_from_directory(
    images_dir,
    labels=None,
    label_mode=None,
    batch_size=32,
    image_size=(256, 256),
    shuffle=False,
)

# Charger les labels à partir du dossier
labels = []
for filename in os.listdir(labels_dir):
    if filename.endswith('.csv'):
        filepath = os.path.join(labels_dir, filename)
        try : 
            label = pd.read_csv(filepath)
            labels.append(label)
        except:
            labels.append(None)
labels_tensor = tf.convert_to_tensor(labels)

# Convertir les images en tenseurs
images_tensor = tf.concat([x for x in images_dataset], axis=0)


Found 703 files belonging to 1 classes.


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 32 * 32, 512)
        self.fc2 = nn.Linear(512, 8)  # 2 classes pour la détection binaire

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 64 * 28 * 28)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [5]:
import tensorflow as tf
from tensorflow.keras import layers

num_classes = 8 
model = tf.keras.Sequential([
    layers.experimental.preprocessing.Rescaling(1./256), 
    layers.Conv2D(128,4, activation='relu'), 
    layers.MaxPooling2D(),
    layers.Conv2D(64,4, activation='relu'), 
    layers.MaxPooling2D(),
    layers.Conv2D(32,4, activation='relu'), 
    layers.MaxPooling2D(),
    layers.Conv2D(16,4, activation='relu'), 
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'), 
    layers.Dense(num_classes, activation='softmax')
])

In [6]:
model.compile(optimizer='adam',loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
data_dir = 'train/images'
data_dir1= 'val/images'

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training" ,
    seed=42,
    image_size=(256, 256))

val_data = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir1,
    validation_split=0.2,
    subset="validation" ,
    seed=42,
    image_size=(256, 256))

Found 0 files belonging to 0 classes.
Using 0 files for training.


ValueError: No images found in directory train/images. Allowed formats: ('.bmp', '.gif', '.jpeg', '.jpg', '.png')

In [13]:
model.fit(
    images_dataset,
    validation_data= images_dataset,
    epochs= 8
)

Epoch 1/8


ValueError: in user code:

    File "/opt/anaconda3/envs/sy32/lib/python3.8/site-packages/keras/src/engine/training.py", line 1338, in train_function  *
        return step_function(self, iterator)
    File "/opt/anaconda3/envs/sy32/lib/python3.8/site-packages/keras/src/engine/training.py", line 1322, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "/opt/anaconda3/envs/sy32/lib/python3.8/site-packages/keras/src/engine/training.py", line 1303, in run_step  **
        outputs = model.train_step(data)
    File "/opt/anaconda3/envs/sy32/lib/python3.8/site-packages/keras/src/engine/training.py", line 1082, in train_step
        self._validate_target_and_loss(y, loss)
    File "/opt/anaconda3/envs/sy32/lib/python3.8/site-packages/keras/src/engine/training.py", line 1036, in _validate_target_and_loss
        raise ValueError(

    ValueError: Target data is missing. Your model was compiled with loss=<keras.src.losses.SparseCategoricalCross

In [54]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

dataset = tf.data.Dataset.from_tensor_slices((images_tensor, labels_tensor))

# Créer le modèle
model = Sequential()

# Ajouter des couches convolutives
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))

# Ajouter des couches denses
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dense(10, activation='softmax'))

# Compiler le modèle
# Définir la fonction de perte et l'optimiseur

# Créer un dataloader
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
# Définir la fonction de perte et l'optimiseur
criterion = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.legacy.Adam()

# Définir le nombre d'époques
num_epochs = 10

# Boucle sur les époques
for epoch in range(num_epochs):
    # Boucle sur les lots
    for images, labels in dataset:
        # Effacer les gradients
        #optimizer.zero_grad()

        # Faire une prédiction
        outputs = model(images)

        # Calculer la perte
        loss = criterion(outputs, labels)

        # Calculer les gradients
        grads = tape.gradient(loss, model.trainable_variables)

        # Mettre à jour les poids
        optimizer.apply_gradients(zip(grads, model.trainable_variables))



# Entraîner le modèle
#images_tensor = np.expand_dims(images_tensor, axis=0)
#model.fit(images_tensor,labels_tensor ,epochs=10)
# Entraîner le modèle
model.fit(images_tensor, labels_tensor, batch_size=32, epochs=10, validation_split=0.2)


AttributeError: 'Sequential' object has no attribute 'train'

In [47]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Définir l'architecture du modèle
model = CNN()

# Définir la fonction de perte et l'optimiseur
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

dataset = tf.data.Dataset.from_tensor_slices((images_tensor, labels_tensor))

# Diviser les données en ensembles d'entraînement et de test
# Par exemple, en utilisant un ratio de 80% pour l'entraînement et 20% pour le test
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])


# Créer des chargeurs de données pour l'entraînement et le test
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Entraîner le modèle
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images_batch, labels_batch in train_dataset:
        optimizer.zero_grad()
        outputs = model(images_batch)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Training Loss: {running_loss / len(train_loader)}")

# Évaluer le modèle sur l'ensemble de test
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images_batch, labels_batch in dataset:
        outputs = model(images_batch)
        _, predicted = torch.max(outputs, 1)
        total += labels_batch.size(0)
        correct += (predicted == labels_batch).sum().item()

print(f"Accuracy on test set: {100 * correct / total}%")


TypeError: '_TensorSliceDataset' object is not subscriptable